# 02 — Data Cleaning & Preprocessing

## What to do
Transform raw data into a clean, structured format ready for machine learning models.

## What you'll learn
- Handle missing values
- Remove duplicates
- Detect and manage outliers
- Encode categorical variables
- Scale numeric features
- Create train-test split
- Build reusable preprocessing pipelines

## Step 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

print('Libraries loaded!')

## Step 2: Load cleaned EDA data

**Action:** Load the dataset from EDA.
**Insight:** Start with the same data we explored in EDA.

In [ ]:
df = pd.read_csv('data/diabetes_prediction_dataset.csv')
print(f'Data shape: {df.shape}')
display(df.head())

## Step 3: Handle missing values

**Action:** Fill or drop missing values based on strategy.
**Insight:** Different strategies work for different columns (numeric: median/mean, categorical: mode/drop).

In [ ]:
# Check missing values
missing = df.isnull().sum()
if missing.sum() > 0:
    print('Missing values found:')
    print(missing[missing > 0])
    
    # Fill numeric columns with median
    numeric_cols = df.select_dtypes(include=['number']).columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    
    # Fill categorical columns with mode
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
else:
    print('✓ No missing values!')

## Step 4: Remove duplicates

**Action:** Identify and drop duplicate rows.
**Insight:** Duplicates can bias model training and inflate accuracy metrics.

In [ ]:
duplicates_before = df.duplicated().sum()
df = df.drop_duplicates()
duplicates_after = df.duplicated().sum()

print(f'Duplicates removed: {duplicates_before}')
print(f'Remaining duplicates: {duplicates_after}')
print(f'New shape: {df.shape}')

## Step 5: Detect and handle outliers

**Action:** Identify outliers using IQR method.
**Insight:** Outliers can distort model training. Options: remove, cap, or keep (depends on domain).

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns

def detect_outliers(df, columns):
    outlier_indices = []
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outlier_indices.extend(df[(df[col] < lower_bound) | (df[col] > upper_bound)].index)
    return list(set(outlier_indices))

outlier_indices = detect_outliers(df, numeric_cols)
print(f'Outliers detected: {len(outlier_indices)}')
print('Note: Keeping outliers as they may be valid. Monitor during modeling.')

## Step 6: Encode categorical variables

**Action:** Convert categorical text to numeric codes.
**Insight:** ML models need numeric input. Use one-hot encoding for nominal, label encoding for ordinal.

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

if categorical_cols:
    print(f'Categorical columns to encode: {categorical_cols}')
    
    # Use one-hot encoding for categorical features
    df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    print(f'\nEncoding complete. New shape: {df_encoded.shape}')
    df = df_encoded
else:
    print('✓ No categorical columns to encode')

## Step 7: Separate features and target

**Action:** Split data into X (features) and y (target).
**Insight:** Target must be isolated for supervised learning.

In [ ]:
target_col = 'diabetes'  # Adjust if different name

X = df.drop(columns=[target_col])
y = df[target_col]

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeature columns: {X.columns.tolist()}')

## Step 8: Train-test split (stratified)

**Action:** Split data into training and testing sets while maintaining target distribution.
**Insight:** Stratified split prevents class imbalance in either set. Test set estimates real-world performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set: {X_test.shape[0]} samples')
print(f'\nTarget distribution in train: {y_train.value_counts().to_dict()}')
print(f'Target distribution in test: {y_test.value_counts().to_dict()}')

## Step 9: Scale numeric features

**Action:** Normalize numeric features to the same scale.
**Insight:** Scaling prevents features with large ranges from dominating. Essential for distance-based algorithms.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on train only
X_test_scaled = scaler.transform(X_test)  # Transform test using train params

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print('✓ Scaling complete!')
print(f'Train mean: {X_train_scaled.mean().mean():.4f}')
print(f'Train std: {X_train_scaled.std().mean():.4f}')

## Step 10: Save preprocessed data

**Action:** Store cleaned data for use in modeling notebooks.
**Insight:** Saves computation time. Ensures consistency across experiments.

In [ ]:
# Save preprocessed data
X_train_scaled.to_csv('data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('data/X_test_scaled.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

print('✓ Preprocessed data saved!')
print('Files:')
print('  - data/X_train_scaled.csv')
print('  - data/X_test_scaled.csv')
print('  - data/y_train.csv')
print('  - data/y_test.csv')

## Preprocessing Summary

**What we did:**
1. Handled missing values (median/mode)
2. Removed duplicates
3. Detected outliers (kept for domain relevance)
4. Encoded categorical variables (one-hot)
5. Split data into train/test (stratified 75/25)
6. Scaled numeric features (StandardScaler)
7. Saved preprocessed data for modeling

**Ready for modeling!**